# 02 - Hyperparameter Tuning (Unified, Fair Harness)

Every model in this study -- the proposed **ELinear** and every deep-learning
benchmark (**LSTM, TCN, NBEATS, DLinear, NLinear, PatchTST, iTransformer**) --
is tuned through the **exact same** rolling-origin CV harness
(`cv_utils.cv_score_generic` + `cv_utils.evaluate_walk_forward`), the same
Optuna TPE sampler, and the **same trial budget**. Only the model-specific
hyperparameter search space differs. This is the core fairness/reviewer-safety
commitment for this study: no model gets a methodological advantage from a
looser validation procedure or a larger search budget.

The 7 deep-learning benchmarks are implemented via **NeuralForecast**
(Garza, Challu & Mergenthaler-Canseco, 2022) rather than a mix of custom
PyTorch code and external CLI tools -- one consistent, auditable
implementation source for every non-proposed model. ELinear remains a
from-scratch implementation (`src/elinear_model.py`) since its training
algorithm (Algorithm 1, mutual learning) is non-standard and is the paper's
actual contribution -- it should be transparent and independently auditable,
not routed through a third-party training loop.

ARIMA is tuned separately via `auto_arima`'s own AICc stepwise search
(Hyndman & Khandakar 2008) -- this is a deliberate, documented exception: order
selection for classical ARIMA is not a hyperparameter search problem in the
same sense as a neural network's, and forcing it through Optuna would be
non-standard.

**Fairness commitments made explicit in this notebook:**
- Identical `N_TRIALS` Bayesian-optimization budget for every DL model.
- Identical `N_FOLDS` rolling-origin CV folds for every DL model.
- Identical `seq_len` search grid for every DL model.
- Identical random seed for the Optuna sampler across every model's study.
- Every model scored by the SAME externally-computed metric function
  (`metrics.mse`), never a library's self-reported training/validation loss.

## Colab setup
Run this once per session. It mounts Google Drive (where the whole `potato_pipeline/` project folder should be uploaded), `cd`s into `notebooks/` so every relative path (`../src`, `../data`, `../results`) keeps working unchanged, and installs the extra packages this notebook needs (Colab's base image doesn't include them).

**GPU:** `Runtime -> Change runtime type -> T4 GPU` recommended for this notebook (deep learning training).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust this if you uploaded the project folder somewhere else in Drive
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q optuna pmdarima neuralforecast

In [ ]:
import sys, os, json, time
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings("ignore")

import cv_utils
import elinear_model
import nf_wrappers
from metrics import mse

optuna.logging.set_verbosity(optuna.logging.WARNING)

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

HORIZONS = [1, 7, 14, 30]
N_TRIALS = 20     # IDENTICAL for every DL model -- fairness commitment
N_FOLDS = 3        # IDENTICAL for every DL model -- fairness commitment
RANDOM_STATE = 42

DL_MODEL_KEYS = ["ELinear", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer"]

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE} | Trial budget per model per horizon: {N_TRIALS} | CV folds: {N_FOLDS}")

## Local-checkpoint-then-sync-to-Drive

Google Drive's FUSE mount can drop mid-session on long-running notebooks (this one runs `N_TRIALS x N_FOLDS x 8 models x 4 horizons` training runs). Results are written to local Colab disk during the loop and copied to Drive once per model, so a transient mount failure costs at most one model's results, not the whole notebook.

In [ ]:
LOCAL_RESULTS = "/content/local_results" if os.path.exists("/content") else "../results_local"
os.makedirs(LOCAL_RESULTS, exist_ok=True)

def safe_sync_to_drive(local_path, drive_path, retries=3, delay=5):
    import shutil
    for attempt in range(retries):
        try:
            shutil.copy(local_path, drive_path)
            return True
        except OSError as e:
            print(f"Drive sync failed (attempt {attempt+1}/{retries}): {e}")
            time.sleep(delay)
    print(f"WARNING: could not sync {local_path} to Drive after {retries} attempts. "
          f"Result is still safe locally at {local_path} -- copy manually later.")
    return False

## 1. Load clean data and hold out the final test block

In [ ]:
df = pd.read_csv("../data/agra_potato_clean.csv", index_col=0, parse_dates=True)
series = df["price"].values
n = len(series)

pool_idx, test_idx = cv_utils.chronological_holdout(n, test_frac=0.15)
series_pool = series[pool_idx]

print(f"Total days: {n} | Tuning pool: {len(series_pool)} | Held-out test days: {len(test_idx)}")

## 2. Unified objective builder

One function builds the Optuna objective for ANY model key -- ELinear or any
NeuralForecast benchmark -- so every model is wired through literally the
same code path, not seven copy-pasted near-duplicates that could silently
drift out of sync with each other.

In [ ]:
def make_objective(model_key, horizon):
    def objective(trial):
        seq_len = trial.suggest_categorical("seq_len", [horizon * m for m in (2, 3, 4, 6)])

        if model_key == "ELinear":
            params = {
                "n_learners": trial.suggest_int("n_learners", 2, 5),
                "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
                "epochs_phase1": trial.suggest_int("epochs_phase1", 30, 100, step=10),
                "epochs_phase2": trial.suggest_int("epochs_phase2", 2, 10),
                "epochs_phase3": trial.suggest_int("epochs_phase3", 10, 40, step=5),
            }
            fit_fn, predict_fn = elinear_model.elinear_fit_fn, elinear_model.elinear_predict_fn
        else:
            common = nf_wrappers.suggest_common_params(trial)
            extra = nf_wrappers.NF_MODEL_REGISTRY[model_key]["search_space"](trial)
            params = {**common, **extra}
            fit_fn = nf_wrappers.make_nf_fit_fn(model_key)
            predict_fn = nf_wrappers.nf_predict_fn

        score = cv_utils.cv_score_generic(
            series_pool, seq_len, horizon, fit_fn, predict_fn, params,
            seed=RANDOM_STATE, n_folds=N_FOLDS, metric_fn=mse,
        )
        trial.set_user_attr("seq_len", seq_len)
        return score
    return objective

## 3. Run the identical tuning loop for every DL model

In [ ]:
all_best_hp = {}
for model_key in DL_MODEL_KEYS:
    best_all = {}
    t0 = time.time()
    for h in HORIZONS:
        study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_STATE))
        study.optimize(make_objective(model_key, h), n_trials=N_TRIALS, show_progress_bar=False)
        best = dict(study.best_params)
        best["seq_len"] = study.best_trial.user_attrs["seq_len"]
        best_all[h] = {"params": best, "val_mse": study.best_value}
        print(f"[{model_key}] horizon={h} best_val_mse={study.best_value:.5f}")

    all_best_hp[model_key] = best_all
    local_path = f"{LOCAL_RESULTS}/best_hp_{model_key.lower()}.json"
    with open(local_path, "w") as f:
        json.dump(best_all, f, indent=2)
    safe_sync_to_drive(local_path, f"{RESULTS_DIR}/best_hp_{model_key.lower()}.json")
    print(f"[{model_key}] done in {time.time()-t0:.1f}s\n")

## 4. ARIMA (documented exception -- see notebook intro)

Daily data with ~365-day annual seasonality is intractable for classical
seasonal ARIMA (m=365). Following Hyndman & Athanasopoulos (*FPP3*, Ch. 10),
weekly structure is modeled directly (m=7) and the annual cycle is captured
via Fourier-term exogenous regressors, with order selection by AICc-based
stepwise search (Hyndman & Khandakar 2008). This ARIMA fit/evaluate loop
also uses `evaluate_walk_forward`-equivalent single-point-ahead scoring for
consistency with every other model's walk-forward evaluation, even though
its "hyperparameter search" is `auto_arima`'s internal AICc search rather
than Optuna.

In [ ]:
import pmdarima as pm
from pmdarima.preprocessing import FourierFeaturizer

def fit_arima(train_series, m=7, k_fourier=3):
    fourier = FourierFeaturizer(m=365, k=k_fourier)
    y_prime, exog = fourier.fit_transform(pd.Series(train_series))
    model = pm.auto_arima(
        y_prime, exogenous=exog,
        seasonal=True, m=m,
        stepwise=True, information_criterion="aicc",
        suppress_warnings=True, error_action="ignore",
    )
    return model, fourier

# ARIMA is fit on z-score normalized data too, for consistency with every
# other model in the study (see fairness commitment in the notebook intro).
# The Fourier calendar terms themselves are unaffected by normalization --
# only the price series (ARIMA's endog) is normalized/inverse-transformed.
arima_best = {}
for h in HORIZONS:
    splits = cv_utils.rolling_origin_splits(len(series_pool), seq_len=1, pred_len=h, n_folds=N_FOLDS)
    fold_scores = []
    for train_idx, val_idx in splits:
        train_end = train_idx[-1] + 1
        mean, std = cv_utils.zscore_stats(series_pool[:train_end])
        train_series = pd.Series(cv_utils.zscore_transform(series_pool[:train_end], mean, std))
        try:
            model, fourier = fit_arima(train_series)
            _, exog_future = fourier.transform(train_series, n_periods=h)
            fc_n = model.predict(n_periods=h, exogenous=exog_future)
            fc = cv_utils.zscore_inverse(np.asarray(fc_n)[:h], mean, std)
            true_val = series_pool[train_end: train_end + h]
            if len(true_val) == h:
                fold_scores.append(mse(true_val, fc))
        except Exception:
            continue
    arima_best[h] = {"val_mse": float(np.mean(fold_scores)) if fold_scores else None}
    print(f"[ARIMA] horizon={h} mean_val_mse={arima_best[h]['val_mse']}")

local_path = f"{LOCAL_RESULTS}/best_hp_arima.json"
with open(local_path, "w") as f:
    json.dump(arima_best, f, indent=2)
safe_sync_to_drive(local_path, f"{RESULTS_DIR}/best_hp_arima.json")

## Summary

All best-hyperparameter JSON files are written to `../results/best_hp_*.json`
(synced from local disk, with retry on Drive mount failures). Notebook 03
loads these to train final models (5 seeds each) on the full tuning pool
before evaluating once on the held-out test block in Notebook 04.

Naive and Seasonal-Naive require no tuning and are implemented directly in
Notebook 04.